# Evaluation — DE-Only vs. Multilingual RAG

Compares the two pipeline configurations side-by-side using:
- **Semantic similarity** to ground truth (cosine similarity via E5 embeddings)
- **LLM-as-judge** scoring (phi4-mini rates each answer 1–5)

> **Prerequisite**: run `mono_rag.ipynb` and `multi_rag.ipynb` first to build both ChromaDB collections.

## Section 0 — Imports & Configuration

In [ ]:
%pip install -q sentence-transformers chromadb rank-bm25 langdetect ollama pandas openpyxl

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import chromadb
import ollama
from sentence_transformers import SentenceTransformer, CrossEncoder

from chunking   import load_documents, build_chunk_records
from retrieval  import build_bm25_index, retrieve
from generation import ask, LANG_NAMES

# ── Two RAG configurations ────────────────────────────────────────────────
CFG_DE_ONLY = {
    "TOP_K":          3,
    "BM25_WEIGHT":    0.4,
    "SEMANTIC_POOL":  30,
    "RERANK_POOL":    6,
    "MODEL_NAME":     "phi4-mini",
    "COLLECTION_NAME": "fdv_de_only",
    "CHROMA_DIR":      "chroma_db_de",
    "DATA_DIRS":       {"de": Path("data/de")},
}

CFG_MULTILINGUAL = {
    "TOP_K":          3,
    "BM25_WEIGHT":    0.4,
    "SEMANTIC_POOL":  30,
    "RERANK_POOL":    6,
    "MODEL_NAME":     "phi4-mini",
    "COLLECTION_NAME": "fdv_multilingual",
    "CHROMA_DIR":      "chroma_db_multilingual",
    "DATA_DIRS":       {"de": Path("data/de"),
                        "fr": Path("data/fr"),
                        "it": Path("data/it")},
}

# ── Scoring flags ─────────────────────────────────────────────────────────
SCORE_SEMANTIC  = True
SCORE_LLM_JUDGE = True

# ── Questions file ────────────────────────────────────────────────────────
QUESTIONS_FILE  = "questions.csv"   # change path if needed

EMBED_MODEL = "intfloat/multilingual-e5-small"
CE_MODEL    = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
BATCH_SIZE  = 64

print("Configuration loaded.")

In [ ]:
# Load shared models (once)
print("Loading embedding model...")
embedder = SentenceTransformer(EMBED_MODEL)

print("Loading cross-encoder...")
cross_encoder = CrossEncoder(CE_MODEL)

def _load_rag_system(cfg):
    """Build chunk_records, BM25 index, and connect to an existing ChromaDB collection."""
    docs = []
    for lang, data_dir in cfg["DATA_DIRS"].items():
        docs.extend(load_documents(data_dir, lang))
    records   = build_chunk_records(docs)
    bm25      = build_bm25_index(records)
    client    = chromadb.PersistentClient(path=str(cfg["CHROMA_DIR"]))
    coll      = client.get_or_create_collection(
        name=cfg["COLLECTION_NAME"],
        metadata={"hnsw:space": "cosine"},
    )
    assert coll.count() > 0, (
        f"Collection '{cfg['COLLECTION_NAME']}' is empty — "
        "run mono_rag.ipynb / multi_rag.ipynb first."
    )
    print(f"  {cfg['COLLECTION_NAME']}: {coll.count()} chunks, {len(records)} records")
    return records, bm25, coll

print("\nLoading DE-only RAG system...")
chunks_de,  bm25_de,  coll_de  = _load_rag_system(CFG_DE_ONLY)
print("Loading multilingual RAG system...")
chunks_ml,  bm25_ml,  coll_ml  = _load_rag_system(CFG_MULTILINGUAL)

## Section 1 — Load Test Questions

In [ ]:
qpath = Path(QUESTIONS_FILE)
if not qpath.exists():
    # Try Excel as fallback
    xlsx = qpath.with_suffix(".xlsx")
    if xlsx.exists():
        df_q = pd.read_excel(xlsx)
        print(f"Loaded from {xlsx}")
    else:
        raise FileNotFoundError(
            f"Questions file not found: {QUESTIONS_FILE}\n"
            "Create a CSV with at least a 'question' column, "
            "optionally 'language' and 'ground_truth' columns."
        )
else:
    df_q = pd.read_csv(qpath)
    print(f"Loaded from {qpath}")

# Normalise column names
df_q.columns = [c.strip().lower() for c in df_q.columns]
assert "question" in df_q.columns, "CSV must have a 'question' column"

if "language"     not in df_q.columns: df_q["language"]     = None
if "ground_truth" not in df_q.columns: df_q["ground_truth"] = None

print(f"{len(df_q)} questions loaded.")
display(df_q.head(10))

## Section 2 — Run Both Setups

In [ ]:
def _top_source(chunks):
    if not chunks:
        return ""
    r   = chunks[0]
    sec = r.get("section_id") or ""
    return f"{r.get('regulation_number','')} §{sec} [{r.get('language','').upper()}]"

results = []
for idx, row in df_q.iterrows():
    q = row["question"]
    print(f"[{idx+1}/{len(df_q)}] {q[:80]}")

    ans_de, cks_de = ask(q, embedder, cross_encoder,
                          coll_de, bm25_de, chunks_de, CFG_DE_ONLY)
    ans_ml, cks_ml = ask(q, embedder, cross_encoder,
                          coll_ml, bm25_ml, chunks_ml, CFG_MULTILINGUAL)

    results.append({
        "question":              q,
        "language":              row.get("language"),
        "ground_truth":          row.get("ground_truth"),
        "answer_de_only":        ans_de,
        "answer_multilingual":   ans_ml,
        "top_source_de_only":    _top_source(cks_de),
        "top_source_multilingual": _top_source(cks_ml),
        "top_rerank_score_de_only":    cks_de[0]["rerank_score"] if cks_de else None,
        "top_rerank_score_multilingual": cks_ml[0]["rerank_score"] if cks_ml else None,
    })

df_results = pd.DataFrame(results)
print("\nDone. Preview:")
display(df_results[["question", "top_source_de_only", "top_source_multilingual"]].head())

## Section 3 — Scoring Configuration

In [ ]:
print(f"SCORE_SEMANTIC  = {SCORE_SEMANTIC}")
print(f"SCORE_LLM_JUDGE = {SCORE_LLM_JUDGE}")

has_gt = df_results["ground_truth"].notna().any()
print(f"Ground truth available: {has_gt}")
if not has_gt:
    print("  → Semantic scoring will compute inter-system agreement (DE-only vs. multilingual).")

## Section 4 — Semantic Similarity Scoring

In [ ]:
if SCORE_SEMANTIC:
    n = len(df_results)

    # Embed answers and ground truths with "query: " prefix (similarity mode)
    all_texts = (
        ["query: " + str(a) for a in df_results["answer_de_only"]] +
        ["query: " + str(a) for a in df_results["answer_multilingual"]] +
        ["query: " + str(gt) if pd.notna(gt) else "query: " for gt in df_results["ground_truth"]]
    )
    all_embs = embedder.encode(all_texts, normalize_embeddings=True, batch_size=BATCH_SIZE)

    embs_de  = all_embs[:n]
    embs_ml  = all_embs[n : 2*n]
    embs_gt  = all_embs[2*n:]

    sem_de_scores = []
    sem_ml_scores = []
    for i in range(n):
        if pd.notna(df_results["ground_truth"].iloc[i]):
            sem_de_scores.append(float(embs_de[i] @ embs_gt[i]))
            sem_ml_scores.append(float(embs_ml[i] @ embs_gt[i]))
        else:
            # No ground truth: use inter-system agreement as proxy
            cross_sim = float(embs_de[i] @ embs_ml[i])
            sem_de_scores.append(cross_sim)
            sem_ml_scores.append(cross_sim)

    df_results["sem_score_de_only"]      = sem_de_scores
    df_results["sem_score_multilingual"] = sem_ml_scores
    df_results["sem_score_delta"]        = [
        ml - de for de, ml in zip(sem_de_scores, sem_ml_scores)
    ]
    print("Semantic scoring complete.")
    print(f"  Avg DE-only sim : {np.mean(sem_de_scores):.4f}")
    print(f"  Avg multilingual: {np.mean(sem_ml_scores):.4f}")
else:
    print("SCORE_SEMANTIC is False — skipping.")

## Section 5 — LLM-as-Judge Scoring

In [ ]:
JUDGE_SYSTEM = (
    "You are an evaluation assistant. Rate how well the RAG answer "
    "matches the ground truth on a scale of 1 to 5, where "
    "5 = fully correct and complete, 1 = completely wrong.\n"
    "Reply with only: SCORE: <1-5>\nREASON: <one sentence>"
)

def _llm_judge(question: str, answer: str, ground_truth=None):
    if pd.isna(ground_truth) or not ground_truth:
        gt_clause = ""
    else:
        gt_clause = f"\nGround truth: {ground_truth}"
    user_msg = f"Question: {question}\nRAG answer: {answer}{gt_clause}"
    try:
        resp = ollama.chat(
            model="phi4-mini",
            messages=[
                {"role": "system", "content": JUDGE_SYSTEM},
                {"role": "user",   "content": user_msg},
            ],
            options={"temperature": 0.0},
        )
        raw = resp.message.content.strip()
        # Extract SCORE: X
        for line in raw.splitlines():
            if line.upper().startswith("SCORE:"):
                return int(line.split(":", 1)[1].strip()[0]), raw
        return None, raw
    except Exception as e:
        return None, str(e)


if SCORE_LLM_JUDGE:
    judge_scores_de  = []
    judge_reasons_de = []
    judge_scores_ml  = []
    judge_reasons_ml = []

    for idx, row in df_results.iterrows():
        print(f"Judging [{idx+1}/{len(df_results)}]...", end="\r")
        score_de, reason_de = _llm_judge(
            row["question"], row["answer_de_only"], row.get("ground_truth"))
        score_ml, reason_ml = _llm_judge(
            row["question"], row["answer_multilingual"], row.get("ground_truth"))
        judge_scores_de.append(score_de)
        judge_reasons_de.append(reason_de)
        judge_scores_ml.append(score_ml)
        judge_reasons_ml.append(reason_ml)

    df_results["llm_score_de_only"]        = judge_scores_de
    df_results["llm_reason_de_only"]       = judge_reasons_de
    df_results["llm_score_multilingual"]   = judge_scores_ml
    df_results["llm_reason_multilingual"]  = judge_reasons_ml

    valid_de = [s for s in judge_scores_de if s is not None]
    valid_ml = [s for s in judge_scores_ml if s is not None]
    print("\nLLM-judge scoring complete.")
    print(f"  Avg DE-only score : {np.mean(valid_de):.2f}" if valid_de else "  No valid DE scores")
    print(f"  Avg multilingual  : {np.mean(valid_ml):.2f}" if valid_ml else "  No valid ML scores")
else:
    print("SCORE_LLM_JUDGE is False — skipping.")

## Section 6 — Per-Question Results Display

In [ ]:
for _, row in df_results.iterrows():
    print("=" * 70)
    print(f"Q: {row['question']}")
    if pd.notna(row.get("ground_truth")):
        print(f"GT: {row['ground_truth']}")

    print(f"\n[Setup 1 — DE only]  source: {row.get('top_source_de_only','')}")
    if SCORE_SEMANTIC and "sem_score_de_only" in df_results.columns:
        print(f"  sem_score: {row.get('sem_score_de_only', 'n/a'):.4f}")
    if SCORE_LLM_JUDGE and "llm_score_de_only" in df_results.columns:
        print(f"  llm_score: {row.get('llm_score_de_only', 'n/a')}")
    print(row['answer_de_only'][:400])

    print(f"\n[Setup 2 — Multilingual]  source: {row.get('top_source_multilingual','')}")
    if SCORE_SEMANTIC and "sem_score_multilingual" in df_results.columns:
        print(f"  sem_score: {row.get('sem_score_multilingual', 'n/a'):.4f}")
    if SCORE_LLM_JUDGE and "llm_score_multilingual" in df_results.columns:
        print(f"  llm_score: {row.get('llm_score_multilingual', 'n/a')}")
    print(row['answer_multilingual'][:400])
    print()

## Section 7 — Summary Table & Export

In [ ]:
summary_data = {
    "System": ["DE-Only", "Multilingual"],
    "Questions": [len(df_results), len(df_results)],
}

if SCORE_SEMANTIC and "sem_score_de_only" in df_results.columns:
    summary_data["Avg Semantic Score"] = [
        round(df_results["sem_score_de_only"].mean(),      4),
        round(df_results["sem_score_multilingual"].mean(), 4),
    ]

if SCORE_LLM_JUDGE and "llm_score_de_only" in df_results.columns:
    summary_data["Avg LLM Judge Score"] = [
        round(pd.to_numeric(df_results["llm_score_de_only"],      errors="coerce").mean(), 2),
        round(pd.to_numeric(df_results["llm_score_multilingual"], errors="coerce").mean(), 2),
    ]

df_summary = pd.DataFrame(summary_data)
display(df_summary)

# Export full results
out_path = Path("evaluation_results.csv")
df_results.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"\nFull results saved to {out_path}")